In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols, logit, probit
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Загрузка данных
loanapp = pd.read_csv('../datasets/loanapp.csv')
SwissLabor = pd.read_csv('../datasets/SwissLabor.csv')
mroz_Greene = pd.read_csv('../datasets/TableF5-1.csv')

# Предобработка категориальных переменных и удаление пропусков
SwissLabor['participation_num'] = SwissLabor['participation'].replace({'yes': 1, 'no': 0}).astype(int)
SwissLabor['foreign'] = SwissLabor['foreign'].replace({'yes': 1, 'no': 0}).astype(int)

mroz_clean = mroz_Greene[['LFP', 'WA', 'FAMINC', 'WE', 'KL6', 'K618', 'CIT', 'UN']].dropna()
loanapp_clean = loanapp[['approve', 'appinc', 'mortno', 'unem', 'dep', 'male']].dropna()
swiss_clean = SwissLabor[['participation_num', 'income', 'age', 'youngkids', 'oldkids']].dropna()

my_digits = 3

# Функция для удобного вывода предельных значений
def print_margeff(res, at='mean', digits=3):
    """
    Извлекает и печатает таблицу предельных значений.
    at='mean' -> Предельные эффекты в средней точке (MEM)
    at='overall' -> Средние предельные эффекты (AME)
    """
    margeff = res.get_margeff(at=at)
    df_me = margeff.summary_frame().round(digits)
    # Переименовываем столбцы для большей схожести со стандартом
    df_me.columns = ['dy/dx', 'Std. Err.', 'z-value', 'p-value', 'CI Lower', 'CI Upper']
    display(Markdown(df_me.to_markdown()))

legend_text = "\n\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"

# Предельные значения 

## для probit

Рассмотрим probit-модель $P(y=1)=\Phi(x'\beta)$, где
$x'\beta=\beta_0+\beta_1x_1+\dots+\beta_kx_k$

Предельные значения $\frac{\partial P(y=1)}{\partial x_j}=\phi(x'\beta)\beta_j$,
где $\phi(z)=\frac{1}{\sqrt{2\pi}}\exp\{-z^2/2\}$

## для logit

Рассмотрим logit-модель $P(y=1)=\Lambda(x'\beta)$, где
$x'\beta=\beta_0+\beta_1x_1+\dots+\beta_kx_k$

Предельные значения $\frac{\partial P(y=1)}{\partial x_j}=\lambda(x'\beta)\beta_j$,
где $\lambda(z)=\frac{\exp(z)}{(1+\exp(z))^2}$

## Средние предельные значения

Обычно рассчитываются 

- предельные значения для каждого регрессора в средней точке (MEM - Marginal Effects at the Mean): 
  * $\phi(\bar{x}'\beta)\beta_j$ для probit 
  * $\lambda(\bar{x}'\beta)\beta_j$ для logit
- среднее по всей выборке предельное значения для каждого регрессора (AME - Average Marginal Effects): 
  * $\overline{\phi(x'\beta)\beta_j}$  для probit 
  * $\overline{\lambda(x'\beta)\beta_j}$ для logit

# labour force equation

Для датасета `TableF5-1` рассмотрим регрессию **LFP на WA, log(FAMINC), WE, KL6, K618, CIT, UN**
трёх спецификаций:

- LPM
- logit
- probit

Результаты подгонки:

In [2]:
#| echo: false
#| warning: false
#| output: asis

f_lfp = 'LFP ~ WA + np.log(FAMINC) + WE + KL6 + K618 + CIT + UN'

regr_LPM_lfp = ols(f_lfp, data=mroz_clean).fit()
regr_logit_lfp = logit(f_lfp, data=mroz_clean).fit(disp=0)
regr_probit_lfp = probit(f_lfp, data=mroz_clean).fit(disp=0)

models_lfp = [regr_LPM_lfp, regr_logit_lfp, regr_probit_lfp]
model_names_lfp = ['LPM', 'logit', 'probit']

compare_table_lfp = summary_col(
    models_lfp, 
    model_names=model_names_lfp,
    stars=True, 
    float_format=f'%0.{my_digits}f'
)

df_comp_lfp = compare_table_lfp.tables[0]

dep_vars_dict_lfp = {col: [m.model.endog_names] for col, m in zip(df_comp_lfp.columns, models_lfp)}
top_row_lfp = pd.DataFrame(dep_vars_dict_lfp, index=['Dep. Variable'])
df_comp_lfp = pd.concat([top_row_lfp, df_comp_lfp])

display(Markdown(df_comp_lfp.to_markdown() + legend_text))

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [3]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_lfp, at='mean', digits=my_digits)

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [4]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_lfp, at='mean', digits=my_digits)

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [5]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_lfp, at='overall', digits=my_digits)

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [6]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_lfp, at='overall', digits=my_digits)

# approve equation

Для датасета `loanapp` рассмотрим регрессию **approve на appinc/100, mortno, unem, dep, male**
трёх спецификаций:

- LPM
- logit
- probit

Результаты подгонки:

In [7]:
#| echo: false
#| warning: false
#| output: asis

f_app = 'approve ~ I(appinc/100) + mortno + unem + dep + male'

regr_LPM_app = ols(f_app, data=loanapp_clean).fit()
regr_logit_app = logit(f_app, data=loanapp_clean).fit(disp=0)
regr_probit_app = probit(f_app, data=loanapp_clean).fit(disp=0)

models_app = [regr_LPM_app, regr_logit_app, regr_probit_app]
model_names_app = ['LPM', 'logit', 'probit']

compare_table_app = summary_col(
    models_app, 
    model_names=model_names_app,
    stars=True, 
    float_format=f'%0.{my_digits}f'
)

df_comp_app = compare_table_app.tables[0]

dep_vars_dict_app = {col: [m.model.endog_names] for col, m in zip(df_comp_app.columns, models_app)}
top_row_app = pd.DataFrame(dep_vars_dict_app, index=['Dep. Variable'])
df_comp_app = pd.concat([top_row_app, df_comp_app])

display(Markdown(df_comp_app.to_markdown() + legend_text))

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [8]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_app, at='mean', digits=my_digits)

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [9]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_app, at='mean', digits=my_digits)

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [10]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_app, at='overall', digits=my_digits)

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [11]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_app, at='overall', digits=my_digits)

# swiss labour force equation

Для датасета `SwissLabour` рассмотрим регрессию **participation на income, age, age^2, youngkids, oldkids**
трёх спецификаций:

- LPM
- logit
- probit

Результаты подгонки:

In [12]:
#| echo: false
#| warning: false
#| output: asis

f_swiss = 'participation_num ~ income + age + I(age**2) + youngkids + oldkids'

regr_LPM_swiss = ols(f_swiss, data=swiss_clean).fit()
regr_logit_swiss = logit(f_swiss, data=swiss_clean).fit(disp=0)
regr_probit_swiss = probit(f_swiss, data=swiss_clean).fit(disp=0)

models_swiss = [regr_LPM_swiss, regr_logit_swiss, regr_probit_swiss]
model_names_swiss = ['LPM', 'logit', 'probit']

compare_table_swiss = summary_col(
    models_swiss, 
    model_names=model_names_swiss,
    stars=True, 
    float_format=f'%0.{my_digits}f'
)

df_comp_swiss = compare_table_swiss.tables[0]

dep_vars_dict_swiss = {col: [m.model.endog_names] for col, m in zip(df_comp_swiss.columns, models_swiss)}
top_row_swiss = pd.DataFrame(dep_vars_dict_swiss, index=['Dep. Variable'])
df_comp_swiss = pd.concat([top_row_swiss, df_comp_swiss])

display(Markdown(df_comp_swiss.to_markdown() + legend_text))

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [13]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_swiss, at='mean', digits=my_digits)

Вычислите предельное значение для каждого регрессора в средней точке (MEM) для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [14]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_swiss, at='mean', digits=my_digits)

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для logit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [15]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_logit_swiss, at='overall', digits=my_digits)

Вычислите среднее по выборке предельное значение (AME) для каждого регрессора для probit модели. 
**Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [16]:
#| echo: false
#| warning: false
#| output: asis

print_margeff(regr_probit_swiss, at='overall', digits=my_digits)